# Section 1: Marker Gene Dotplot

## Purpose
Build marker-gene dot plots and export the shared color palettes (sub-cell-type, broad-cell-type, niche, stage) used across the manuscript figures.

## Workflow Overview


In [ ]:
%load_ext autoreload
%autoreload 2

## Setup


In [ ]:
# System utilities
import os
import pickle
from pathlib import Path
from datetime import datetime
import warnings
import time
import math

# Data handling and numerical computation
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib_venn import venn3
import matplotlib.colors as mcolors
from upsetplot import UpSet, from_memberships
from plot_utils import proportion
from utils import plot_upset
from utils import scatter_df
import matplotlib.ticker as ticker

# Single-cell analysis and related packages
import anndata as ad
import scanpy as sc
import squidpy as sq

# Suppress warnings
warnings.filterwarnings('ignore', category=FutureWarning, module='numpy')
warnings.filterwarnings('ignore', category=FutureWarning, module='scanpy')
warnings.filterwarnings('ignore', category=UserWarning, module='scanpy')
warnings.filterwarnings('ignore', category=UserWarning, module='numpy')
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Performance
from joblib import Parallel, delayed

# Function to print the current time with a message
def print_with_time(message):
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {message}")

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

plt.rcParams['pdf.fonttype'] = 42  # Ensures text is stored as text, not paths
plt.rcParams['ps.fonttype'] = 42


!pip install upsetplot

pip install numpy scanpy squidpy pandas scikit-learn matplotlib seaborn --upgrade --force

In [ ]:
# Imports: load spatial analysis, plotting, and utility dependencies for spacetime profiling
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon
from shapely.geometry import Polygon
from shapely.ops import unary_union

def convert2gpd(df):
    # Step 1: Group the DataFrame by 'cell_id' or similar, assuming each cell has a unique ID
    # Ensure that your DataFrame has an identifier for each cell
    grouped = df.groupby('cell_id')

    # Step 2: Create polygons for each group of cell boundaries
    polygons = []

    for cell_id, group in grouped:
        # Extract the x and y coordinates for this cell
        points = group[['vertex_x', 'vertex_y']].values
        
        # Create a Polygon from these points
        # Ensure the points form a valid polygon (e.g., no crossing lines)
        if len(points) > 2:  # A polygon needs at least 3 points
            poly = Polygon(points)
            polygons.append({'cell_id': cell_id, 'geometry': poly})

    # Step 3: Create a GeoPandas DataFrame from the list of polygons
    gdf = gpd.GeoDataFrame(polygons)
    return gdf

## Plotting Utilities and Output Paths


In [ ]:

# Helper plotting utility: export color dictionaries used across spacetime figures
def plot_color_palette(color_dict, title, pdf):
    labels = list(color_dict.keys())
    colors = list(color_dict.values())
    num_colors = len(colors)

    # Create a figure and a set of subplots
    fig, ax = plt.subplots(figsize=(5, num_colors // 2))

    # Plot each color as a horizontal bar
    for i, (label, color) in enumerate(zip(labels, colors)):
        ax.barh(i, 1, color=color)
        ax.text(0.5, i, label, va='center', ha='center', fontsize=10, color='white', fontweight='bold')

    # Remove axes
    ax.set_xlim(0, 1)
    ax.set_xticks([])
    ax.set_yticks([])

    # Set title
    ax.set_title(title, fontweight='bold')

    # Save the current figure to the PDF
    if pdf:
        pdf.savefig(fig)
    plt.show()
    plt.close(fig)

## Data Loading


In [ ]:
# Process samples
samples = ['KS_TMA_1_0026870', 'KS_TMA_2_0026882', 'KS_TMA_3_0027198', 'KS_TMA_4_0026764', 'KS_TMA_5_0026776',
           'KS_TMA_6_0027092', 'KS_TMA_7_0027079', 'KS_TMA_8_0027273', 'KS_TMA_9_0026831', 'KS_TMA_10_0026828',
           'KS_TMA_11_0026930', 'KS_TMA_12_0026888', 'KS_TMA_13_0027077', 'KS_TMA_14_0027019', 'KS_TMA_15_0033811',
          'KS_TMA_16_0033809']

marker_list_df_all = pd.read_csv('../data/marker_list_dev_standardized_short.csv')
marker_list_df = marker_list_df_all.copy()
marker_list_df = marker_list_df.query(f"Annotation not in ['Vascular Endothelial Cells', 'Lymphatic Endothelial Cells']")

# Define cell types and clusters
cell_types = list(marker_list_df.keys())

marker_list = marker_list_df
marker_list = marker_list.groupby('grouped_cts')['Gene'].unique().reset_index()
marker_list = marker_list.set_index('grouped_cts')['Gene'].apply(list).to_dict()

complete_cell_types = list(marker_list_df['grouped_cts'].unique())
marker_genes_dict = marker_list

In [ ]:
# Process samples
samples = ['KS_TMA_1_0026870', 'KS_TMA_2_0026882', 'KS_TMA_3_0027198', 'KS_TMA_4_0026764', 'KS_TMA_5_0026776',
           'KS_TMA_6_0027092', 'KS_TMA_7_0027079', 'KS_TMA_8_0027273', 'KS_TMA_9_0026831', 'KS_TMA_10_0026828', 
           'KS_TMA_11_0026930', 'KS_TMA_12_0026888', 'KS_TMA_13_0027077', 'KS_TMA_14_0027019', 'KS_TMA_15_0033811',
          'KS_TMA_16_0033809']

adata = sc.read_h5ad('../data/KS_adata_preprocessed.h5ad')

adata.obs['niches'] = adata.obs.niche_with_tumor_proximity.copy()


adata.obsm["spatial"] = adata.obs[["local_x", "local_y"]].copy().to_numpy()

In [ ]:
adata

In [ ]:
# Define viral marker sets and shared annotation color dictionaries
KS_lytic_genes = ['KSHV.ORF50', 'KSHV.ORF57', 'KSHV.ORF59', 'KSHV.K9', 'KSHV.ORF65'] 
KS_latent_genes = ['KSHV.ORF71','KSHV.ORF72','KSHV.ORF73',] 
KS_K2_gene = ['KSHV.K2']

nctc_neighbors = 30
n_clusters = 10

In [ ]:
# Define color mappings for subtypes, broad cell types, and niche groupings
sub_cell_types_color_mapping = {
    'Keratinocytes': '#181c82',
    'Differentiated Keratinocytes': '#471fc7',
    'Spinous to Granular Cells': '#034cff',
    'Pilosebaceous Cells': '#bbbde2',
    'Melanocytes': '#00bbbf',

    'Vascular Endothelial Cells': '#a4e000',
    'Lymphatic Endothelial Cells': '#ffb695',
    'Proliferating Lymphatic Endothelial Cells': '#906855',
    'Pericytes': '#9f7704',
    
    'Fibroblasts': '#c7d0c0',
    'Pro-inflammatory Fibroblasts': '#7cd28e',
    'Mesenchymal Fibroblasts': '#3d8e27',
    'Myofibroblasts': '#007c1d',
    'Secretory-papillary Fibroblasts': '#076018',
    'Secretory-reticular Fibroblasts': '#324708',
    
    'Macrophages': '#ff40ff',
    'Dendritic cells': '#ff9300',
    'B-cells': '#f12d00',
    'T-cells': '#941100',
    'Cd4': '#600c09',
    'Cd4 Rgcc': '#3c0c09',
    'Cd8 Exhausted': '#240c09',
}


with PdfPages('../figures/sub_cell_types_color_mapping.pdf') as pdf:
    plot_color_palette(sub_cell_types_color_mapping, "Cell Subtypes", pdf)

In [ ]:
# Define color mappings for subtypes, broad cell types, and niche groupings
broad_cell_types_color_mapping = {\
    'Lymphatic Endothelial Cells': '#ffb695',
    'Macrophages': '#ff40ff',
    'Vascular Endothelial Cells': '#a4e000',
    'Pericytes': '#9f7704',
    'Fibroblasts': '#c7d0c0',
    'T-cells': '#941100',
    'Keratinocytes': '#181c82',
    'Dendritic cells': '#ff9300',
    'Spinous to Granular Cells': '#034cff',
    'Pilosebaceous Cells': '#bbbde2',
    'B-cells': '#f12d00',
    'Melanocytes': '#00bbbf'
}

with PdfPages('../figures/broad_cell_types_color_mapping.pdf') as pdf:
    plot_color_palette(broad_cell_types_color_mapping, "Broad Cell Types", pdf)

In [ ]:
# Define color mappings for subtypes, broad cell types, and niche groupings
niche_colors = {
    #SKIN
    "Basal Dermis": "#b299e3",
    "Differentiated Epidermis": "#ffd000",
 
    # STROMA
    "Stroma": "#646500",
    "TA VEC Stroma": "#00e50c",
    "nTA VEC Stroma": "#cccc33",
    
    # IMMUNE
    "Macrophage Immune Stroma": "#00dbf4",
    "T-cell Immune Stroma": "#0051f9",
    "Immune": "#c100f9",
 
    # TUMOR
    "Tumor Core": "#450000",
    "Tumor": "#eb0000",
    "Tumor Boundary": "#faa0aa"
}


with PdfPages('../figures/niches_color_mapping.pdf') as pdf:
    plot_color_palette(niche_colors, "Niches", pdf)

In [ ]:

# Define colors for different stages
stage_colors = {
    'nodular': '#ed322f',  # Light Red
    'plaque': '#ffdc5e',   # Light Yellow
    'patch': '#51f512',    # Light Green
    'control': '#D3D3D3'   # Light Gray
}


niche_colors = {
    #SKIN
    "Basal Dermis": "#b299e3",
    "Differentiated Epidermis": "#ffd000",
 
    # STROMA
    "Stroma": "#646500",
    "TA VEC Stroma": "#00e50c",
    "nTA VEC Stroma": "#cccc33",
    
    # IMMUNE
    "Macrophage Immune Stroma": "#00dbf4",
    "T-cell Immune Stroma": "#0051f9",
    "Immune": "#c100f9",
 
    # TUMOR
    "Tumor Core": "#450000",
    "Tumor": "#eb0000",
    "Tumor Boundary": "#faa0aa"
}


broad_cell_types_color_mapping = {\
    'Lymphatic Endothelial Cells': '#ffb695',
    'Macrophages': '#ff40ff',
    'Vascular Endothelial Cells': '#a4e000',
    'Pericytes': '#9f7704',
    'Fibroblasts': '#c7d0c0',
    'T-cells': '#941100',
    'Keratinocytes': '#181c82',
    'Dendritic cells': '#ff9300',
    'Spinous to Granular Cells': '#034cff',
    'Pilosebaceous Cells': '#bbbde2',
    'B-cells': '#f12d00',
    'Melanocytes': '#00bbbf'
}



In [ ]:
marker_genes_dict['Pilosebaceous Cells']

In [ ]:
adata.obs.broad_cell_types.unique().tolist()

In [ ]:
import scanpy as sc

# Marker gene dictionary keyed by abbreviation (matches the figure)
marker_genes = {
    "LEC":  ["LYVE1", "PROX1"],
    "VEC":  ["PLVAP", "SOX17", "SELE", "SPARCL1"],
    "Fb":   ["LUM", "PDGFRA", "POSTN", "SFRP2"],
    "PC":   ["ACTA2", "PDGFRB", "RGS5"],
    "StoG": ["KRT2"],
    "PSC":  ["CRABP2", "DEFB1", "TMEM45A"],
    "MC":   ["DCT", "TYRP1", "MLANA", "QPCT"],
    "KC":   ["KRT5", "KRT15"],
    "B":    ["CD79A", "TNFRSF17"],
    "T":    ["CD3D", "CD3E", "CD3G"],
    "Mφ":   ["CD68", "FCER1G", "C1QA"],
    "DC":   ["IDO1", "IRF8", "WDFY4", "CLEC9A"],
}

# Map full names → abbreviations
abbrev_map = {
    "Lymphatic Endothelial Cells": "LEC",
    "Vascular Endothelial Cells":  "VEC",
    "Fibroblasts":                 "Fb",
    "Pericytes":                   "PC",
    "Spinous to Granular Cells":   "StoG",
    "Pilosebaceous Cells":         "PSC",
    "Melanocytes":                 "MC",
    "Keratinocytes":               "KC",
    "B-cells":                     "B",
    "T-cells":                     "T",
    "Macrophages":                 "Mφ",
    "Dendritic cells":             "DC",
}

# Add an abbreviated column to obs
adata.obs["cell_type_abbrev"] = (
    adata.obs["broad_cell_types"].map(abbrev_map).astype("category")
)

# Enforce row order to match the figure (top → bottom)
category_order = ["LEC", "VEC", "Fb", "PC", "StoG", "PSC",
                  "MC", "KC", "B", "T", "Mφ", "DC"]

adata.obs["cell_type_abbrev"] = adata.obs["cell_type_abbrev"].cat.reorder_categories(
    category_order
)

sc.pl.dotplot(
    adata,
    var_names=marker_genes,
    groupby="cell_type_abbrev",
    standard_scale="var",
    cmap="Reds",
    dot_max=1.0,
    dot_min=0.0,
    var_group_rotation=0,
    figsize=(14, 5),
    swap_axes=False,
    categories_order=category_order, save='dotplot'
)